<a href="https://colab.research.google.com/github/tawsif113/privacy-utility-dp-ids/blob/main/notebooks/04_dp_sgd_feasibility_smoke_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 04 — DP-SGD Feasibility and PyTorch Parity Smoke Test

## Purpose

This notebook validates the technical foundation for formal DP-SGD before any privacy–utility sweep.

It performs four checks:

1. Reconstruct the accepted locked split and fixed preprocessing from Experiments 02–03.
2. Rebuild the target MLP in PyTorch and test utility parity against the accepted scikit-learn baseline.
3. Verify that the PyTorch model is compatible with Opacus.
4. Run one formally accounted DP-SGD smoke configuration and obtain an actual \((\varepsilon, \delta)\).

## Scope boundary

- This is a **feasibility test**, not the final DP-SGD experiment.
- No MIA is run here.
- The DP configuration is not selected as an optimal privacy–utility point.
- The reported privacy guarantee applies to DP-SGD training **conditional on the already-fixed preprocessing**.


## 1. Install and import dependencies

In [6]:
# Install Opacus only when it is unavailable in the current Colab runtime.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("opacus") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "opacus",
    ])

print("Opacus is available.")

Opacus is available.


In [7]:
# Imports
from __future__ import annotations

import copy
import hashlib
import json
import os
import platform
import random
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 100)

print({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "opacus": __import__("opacus").__version__,
    "sklearn": sklearn.__version__,
})

{'python': '3.12.13', 'torch': '2.11.0+cu128', 'opacus': '1.6.0', 'sklearn': '1.6.1'}


## 2. Fixed experiment configuration

In [29]:
# Reproducibility
SEED = 42

# Architecture and non-private parity training
HIDDEN_DIMS = (64, 32)
NON_PRIVATE_EPOCHS = 30
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# One technical DP-SGD smoke configuration
DP_EPOCHS = 5
TARGET_EPSILON = 8.0
MAX_GRAD_NORM = 1.0
ACCOUNTANT = "prv"
SECURE_MODE = False
POISSON_SAMPLING = True

# PyTorch parity gate against the accepted scikit-learn target MLP
PARITY_TOLERANCES = {
    "f1": 0.04,
    "recall": 0.05,
    "fnr": 0.05,
    "pr_auc": 0.04,
}

def set_all_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.use_deterministic_algorithms(
        True,
        warn_only=True,
    )

set_all_seeds(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print({
    "device": str(DEVICE),
    "non_private_epochs": NON_PRIVATE_EPOCHS,
    "dp_epochs": DP_EPOCHS,
    "target_epsilon": TARGET_EPSILON,
    "accountant": ACCOUNTANT,
})

{'device': 'cuda', 'non_private_epochs': 30, 'dp_epochs': 5, 'target_epsilon': 8.0, 'accountant': 'prv'}


## 3. Mount Drive and resolve canonical Experiment 02–03 artifacts

In [9]:
# Mount Google Drive when running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DEFAULT_PROJECT_DIR = Path(
    "/content/drive/MyDrive/ML-DP-NID"
)
PROJECT_DIR = Path(
    os.environ.get(
        "ML_DP_NID_DIR",
        DEFAULT_PROJECT_DIR,
    )
)

if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()

TRAIN_FILE = PROJECT_DIR / "KDDTrain+.txt"
TEST_FILE = PROJECT_DIR / "KDDTest+.txt"

SPLIT_DIR = PROJECT_DIR / "data" / "split_indices"
TARGET_TRAIN_INDEX_FILE = (
    SPLIT_DIR / "target_train_indices.npy"
)
TARGET_VALIDATION_INDEX_FILE = (
    SPLIT_DIR / "target_validation_indices.npy"
)
SHADOW_POOL_INDEX_FILE = (
    SPLIT_DIR / "shadow_pool_indices.npy"
)

PREPROCESSOR_FILE = (
    PROJECT_DIR
    / "artifacts"
    / "preprocessor"
    / "target_mlp_preprocessor.joblib"
)
BASELINE_MANIFEST_FILE = (
    PROJECT_DIR
    / "artifacts"
    / "manifests"
    / "baseline_mia_combined_manifest.json"
)
BASELINE_RESULTS_FILE = (
    PROJECT_DIR
    / "results"
    / "baseline_mia_combined"
    / "target_ids_results.csv"
)

RESULTS_DIR = (
    PROJECT_DIR
    / "results"
    / "dp_sgd_smoke_test"
)
MODEL_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "models"
)
MANIFEST_DIR = (
    PROJECT_DIR
    / "artifacts"
    / "manifests"
)

for directory in [
    RESULTS_DIR,
    MODEL_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

required_files = [
    TRAIN_FILE,
    TEST_FILE,
    TARGET_TRAIN_INDEX_FILE,
    TARGET_VALIDATION_INDEX_FILE,
    SHADOW_POOL_INDEX_FILE,
    PREPROCESSOR_FILE,
    BASELINE_MANIFEST_FILE,
    BASELINE_RESULTS_FILE,
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

assert not missing_files, (
    "Missing required Experiment 02–03 files:\n"
    + "\n".join(missing_files)
)

print("Project directory:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/ML-DP-NID


## 4. Verify the accepted manifest, datasets, and locked split

In [10]:
# NSL-KDD schema
COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins",
    "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files",
    "num_outbound_cmds", "is_host_login", "is_guest_login", "count",
    "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "label", "difficulty",
]
FEATURES = COLUMNS[:41]

def calculate_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()

def load_nsl_kdd(path: Path) -> pd.DataFrame:
    dataframe = pd.read_csv(
        path,
        names=COLUMNS,
    )

    dataframe["label_clean"] = (
        dataframe["label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.rstrip(".")
    )
    dataframe["binary_label"] = (
        dataframe["label_clean"] != "normal"
    ).astype(np.float32)

    return dataframe

In [11]:
# Load and validate the accepted manifest.
with BASELINE_MANIFEST_FILE.open("r") as file:
    baseline_manifest = json.load(file)

assert (
    baseline_manifest["protocol_version"]
    == "ROADMAP_REVISED_V2"
)
assert baseline_manifest["run_mode"] == "final"
assert (
    baseline_manifest["number_of_shadow_models"]
    == 5
)

actual_train_hash = calculate_sha256(TRAIN_FILE)
actual_test_hash = calculate_sha256(TEST_FILE)

assert (
    actual_train_hash
    == baseline_manifest["dataset"]["train_sha256"]
), "KDDTrain+ hash does not match Experiment 02–03."

assert (
    actual_test_hash
    == baseline_manifest["dataset"]["test_sha256"]
), "KDDTest+ hash does not match Experiment 02–03."

print("Dataset hashes match the accepted manifest.")

Dataset hashes match the accepted manifest.


In [12]:
# Load raw datasets and locked indices.
train_df = load_nsl_kdd(TRAIN_FILE)
test_df = load_nsl_kdd(TEST_FILE)

target_train_indices = np.load(
    TARGET_TRAIN_INDEX_FILE
)
target_validation_indices = np.load(
    TARGET_VALIDATION_INDEX_FILE
)
shadow_pool_indices = np.load(
    SHADOW_POOL_INDEX_FILE
)

assert len(target_train_indices) == 88181
assert len(target_validation_indices) == 12597
assert len(shadow_pool_indices) == 25195

all_indices = np.concatenate([
    target_train_indices,
    target_validation_indices,
    shadow_pool_indices,
])

assert len(np.unique(all_indices)) == len(train_df)
assert set(all_indices) == set(range(len(train_df)))

target_train = (
    train_df.iloc[target_train_indices]
    .copy()
    .reset_index(drop=True)
)
target_validation = (
    train_df.iloc[target_validation_indices]
    .copy()
    .reset_index(drop=True)
)

print({
    "target_train": len(target_train),
    "target_validation": len(target_validation),
    "KDDTest+": len(test_df),
})

{'target_train': 88181, 'target_validation': 12597, 'KDDTest+': 22544}


## 5. Load the fixed preprocessor and create tensors

In [13]:
# Reuse the exact target preprocessor fitted in Experiment 02–03.
target_preprocessor = joblib.load(
    PREPROCESSOR_FILE
)

X_train = target_preprocessor.transform(
    target_train[FEATURES]
)
X_validation = target_preprocessor.transform(
    target_validation[FEATURES]
)
X_test = target_preprocessor.transform(
    test_df[FEATURES]
)

# Be robust to either dense or sparse transformer outputs.
if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
    X_validation = X_validation.toarray()
    X_test = X_test.toarray()

X_train = np.asarray(
    X_train,
    dtype=np.float32,
)
X_validation = np.asarray(
    X_validation,
    dtype=np.float32,
)
X_test = np.asarray(
    X_test,
    dtype=np.float32,
)

y_train = target_train[
    "binary_label"
].to_numpy(dtype=np.float32).reshape(-1, 1)

y_validation = target_validation[
    "binary_label"
].to_numpy(dtype=np.float32).reshape(-1, 1)

y_test = test_df[
    "binary_label"
].to_numpy(dtype=np.float32).reshape(-1, 1)

INPUT_DIM = X_train.shape[1]
DELTA = 1.0 / len(X_train)

print({
    "input_dim": INPUT_DIM,
    "delta": DELTA,
    "train_shape": X_train.shape,
})

{'input_dim': 122, 'delta': 1.134031140495118e-05, 'train_shape': (88181, 122)}


In [14]:
# Tensor datasets and deterministic data-loader factory
train_dataset = TensorDataset(
    torch.from_numpy(X_train),
    torch.from_numpy(y_train),
)
validation_dataset = TensorDataset(
    torch.from_numpy(X_validation),
    torch.from_numpy(y_validation),
)
test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(y_test),
)

def create_train_loader(seed: int) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
        drop_last=False,
    )

validation_loader = DataLoader(
    validation_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1024,
    shuffle=False,
    num_workers=0,
)

## 6. Define the Opacus-compatible PyTorch MLP

In [15]:
class BinaryMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: tuple[int, int] = HIDDEN_DIMS,
    ) -> None:
        super().__init__()

        hidden_1, hidden_2 = hidden_dims

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_1),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Linear(hidden_2, 1),
        )

    def forward(
        self,
        features: torch.Tensor,
    ) -> torch.Tensor:
        return self.network(features)

set_all_seeds(SEED)

initial_model = BinaryMLP(INPUT_DIM)
initial_state_dict = copy.deepcopy(
    initial_model.state_dict()
)

compatibility_errors = ModuleValidator.validate(
    initial_model,
    strict=False,
)

contains_batch_norm = any(
    isinstance(
        module,
        (
            nn.BatchNorm1d,
            nn.BatchNorm2d,
            nn.BatchNorm3d,
        ),
    )
    for module in initial_model.modules()
)

assert not contains_batch_norm
assert not compatibility_errors, (
    "Model is not Opacus-compatible: "
    + str(compatibility_errors)
)

print(initial_model)
print("Opacus compatibility check passed.")

BinaryMLP(
  (network): Sequential(
    (0): Linear(in_features=122, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)
Opacus compatibility check passed.


## 7. Shared training, prediction, threshold, and metric helpers

In [16]:
def train_one_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> float:
    model.train()

    total_weighted_loss = 0.0
    total_examples = 0

    for features, labels in data_loader:
        features = features.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(features)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        current_batch_size = labels.shape[0]
        total_weighted_loss += (
            float(loss.detach().cpu())
            * current_batch_size
        )
        total_examples += current_batch_size

    return total_weighted_loss / max(
        total_examples,
        1,
    )

@torch.no_grad()
def predict_probabilities(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    probability_batches = []

    for features, _ in data_loader:
        features = features.to(
            device,
            non_blocking=True,
        )

        logits = model(features)
        probabilities = torch.sigmoid(logits)

        probability_batches.append(
            probabilities.cpu().numpy()
        )

    return np.concatenate(
        probability_batches,
        axis=0,
    ).reshape(-1)

In [17]:
def select_f2_threshold(
    y_true: np.ndarray,
    probabilities: np.ndarray,
) -> tuple[float, pd.DataFrame]:
    rows = []

    for threshold in np.arange(0.01, 1.00, 0.01):
        predictions = (
            probabilities >= threshold
        ).astype(int)

        rows.append({
            "threshold": float(threshold),
            "f2": fbeta_score(
                y_true,
                predictions,
                beta=2,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                predictions,
                zero_division=0,
            ),
            "f1": f1_score(
                y_true,
                predictions,
                zero_division=0,
            ),
        })

    search = pd.DataFrame(rows)

    selected_threshold = float(
        search.sort_values(
            ["f2", "recall"],
            ascending=False,
        )
        .iloc[0]["threshold"]
    )

    return selected_threshold, search

def calculate_ids_metrics(
    model_name: str,
    split_name: str,
    threshold_policy: str,
    y_true: np.ndarray,
    probabilities: np.ndarray,
    threshold: float,
) -> dict:
    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()

    return {
        "model": model_name,
        "split": split_name,
        "threshold_policy": threshold_policy,
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "fnr": fn / (fn + tp) if fn + tp else np.nan,
        "fpr": fp / (fp + tn) if fp + tn else np.nan,
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

## 8. Train the non-private PyTorch MLP

In [18]:
# Start from the fixed shared initialization.
set_all_seeds(SEED)

non_private_model = BinaryMLP(INPUT_DIM)
non_private_model.load_state_dict(initial_state_dict)
non_private_model = non_private_model.to(DEVICE)

non_private_optimizer = torch.optim.Adam(
    non_private_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
criterion = nn.BCEWithLogitsLoss()

non_private_train_loader = create_train_loader(seed=SEED)

non_private_history_rows = []
non_private_started_at = time.time()

for epoch in range(1, NON_PRIVATE_EPOCHS + 1):
    epoch_loss = train_one_epoch(
        non_private_model,
        non_private_train_loader,
        non_private_optimizer,
        criterion,
        DEVICE,
    )

    non_private_history_rows.append({
        "epoch": epoch,
        "train_loss": epoch_loss,
    })

    if epoch == 1 or epoch % 5 == 0 or epoch == NON_PRIVATE_EPOCHS:
        print(
            f"Non-private epoch {epoch:02d}/{NON_PRIVATE_EPOCHS}: "
            f"loss={epoch_loss:.6f}"
        )

non_private_training_seconds = time.time() - non_private_started_at
non_private_history = pd.DataFrame(non_private_history_rows)
non_private_history.to_csv(
    RESULTS_DIR / "non_private_pytorch_training_history.csv",
    index=False,
)

print("Non-private training seconds:", round(non_private_training_seconds, 2))

Non-private epoch 01/30: loss=0.157739
Non-private epoch 05/30: loss=0.037549
Non-private epoch 10/30: loss=0.024811
Non-private epoch 15/30: loss=0.018513
Non-private epoch 20/30: loss=0.015758
Non-private epoch 25/30: loss=0.014059
Non-private epoch 30/30: loss=0.013386
Non-private training seconds: 37.06


## 9. Evaluate non-private PyTorch utility and enforce parity

In [19]:
non_private_validation_probabilities = predict_probabilities(
    non_private_model,
    validation_loader,
    DEVICE,
)
non_private_test_probabilities = predict_probabilities(
    non_private_model,
    test_loader,
    DEVICE,
)

validation_labels = y_validation.reshape(-1).astype(int)
test_labels = y_test.reshape(-1).astype(int)

non_private_selected_threshold, non_private_threshold_search = select_f2_threshold(
    validation_labels,
    non_private_validation_probabilities,
)

non_private_results = pd.DataFrame([
    calculate_ids_metrics(
        "pytorch_non_private",
        "target_validation",
        "default_0_5",
        validation_labels,
        non_private_validation_probabilities,
        0.5,
    ),
    calculate_ids_metrics(
        "pytorch_non_private",
        "KDDTest+",
        "default_0_5",
        test_labels,
        non_private_test_probabilities,
        0.5,
    ),
    calculate_ids_metrics(
        "pytorch_non_private",
        "KDDTest+",
        "validation_selected_F2",
        test_labels,
        non_private_test_probabilities,
        non_private_selected_threshold,
    ),
])

non_private_threshold_search.to_csv(
    RESULTS_DIR / "non_private_pytorch_threshold_search.csv",
    index=False,
)
non_private_results.to_csv(
    RESULTS_DIR / "non_private_pytorch_ids_results.csv",
    index=False,
)

display(non_private_results)

,model,split,threshold_policy,threshold,accuracy,precision,recall,f1,fnr,fpr,roc_auc,pr_auc,tn,fp,fn,tp
0,pytorch_non_private,target_validation,default_0_5,0.50,0.994840,0.994204,0.994713,0.994458,0.005287,0.005049,0.999817,0.999800,6700,34,31,5832
1,pytorch_non_private,KDDTest+,default_0_5,0.50,0.810726,0.963528,0.693758,0.806687,0.306242,0.034703,0.910364,0.935714,9374,337,3930,8903
2,pytorch_non_private,KDDTest+,validation_selected_F2,0.29,0.816537,0.958945,0.708018,0.814596,0.291982,0.040058,0.910364,0.935714,9322,389,3747,9086


In [20]:
baseline_results = pd.read_csv(BASELINE_RESULTS_FILE)

baseline_tuned = baseline_results[
    (baseline_results["split"] == "KDDTest+")
    & (baseline_results["threshold_policy"] == "validation_selected_F2")
].iloc[0]

pytorch_tuned = non_private_results[
    (non_private_results["split"] == "KDDTest+")
    & (non_private_results["threshold_policy"] == "validation_selected_F2")
].iloc[0]

parity_rows = []

for metric, tolerance in PARITY_TOLERANCES.items():
    baseline_value = float(baseline_tuned[metric])
    pytorch_value = float(pytorch_tuned[metric])
    absolute_difference = abs(pytorch_value - baseline_value)

    parity_rows.append({
        "metric": metric,
        "sklearn_baseline": baseline_value,
        "pytorch_non_private": pytorch_value,
        "absolute_difference": absolute_difference,
        "tolerance": tolerance,
        "passes": absolute_difference <= tolerance,
    })

parity_results = pd.DataFrame(parity_rows)
parity_pass = bool(parity_results["passes"].all())

parity_results.to_csv(
    RESULTS_DIR / "non_private_pytorch_parity.csv",
    index=False,
)

display(parity_results)
print("Parity gate passed:", parity_pass)

if not parity_pass:
    raise RuntimeError(
        "PyTorch parity gate failed. Do not run DP-SGD until the differences are reviewed and corrected."
    )

,metric,sklearn_baseline,pytorch_non_private,absolute_difference,tolerance,passes
0,f1,0.802027,0.814596,0.012569,0.04,True
1,recall,0.709187,0.708018,0.001169,0.05,True
2,fnr,0.290813,0.291982,0.001169,0.05,True
3,pr_auc,0.917065,0.935714,0.018649,0.04,True


Parity gate passed: True


## 10. Configure the formal DP-SGD smoke run

The private model starts from the **same initial weights** as the non-private PyTorch model.

The training loader is passed through Opacus with Poisson sampling. Opacus then:

- calculates per-sample gradients;
- clips each sample's gradient to `MAX_GRAD_NORM`;
- adds Gaussian noise;
- accounts for the spent privacy budget.

This notebook asks Opacus to choose a noise multiplier intended to reach the target epsilon after the configured number of DP epochs.


In [21]:
set_all_seeds(SEED)

dp_model = BinaryMLP(INPUT_DIM)
dp_model.load_state_dict(initial_state_dict)
dp_model = dp_model.to(DEVICE)

dp_optimizer = torch.optim.Adam(
    dp_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

dp_train_loader = create_train_loader(seed=SEED)

dp_compatibility_errors = ModuleValidator.validate(dp_model, strict=False)
assert not dp_compatibility_errors, (
    "DP model compatibility failure: " + str(dp_compatibility_errors)
)

privacy_engine = PrivacyEngine(
    accountant=ACCOUNTANT,
    secure_mode=SECURE_MODE,
)

is_compatible = privacy_engine.is_compatible(
    module=dp_model,
    optimizer=dp_optimizer,
    data_loader=dp_train_loader,
)

assert is_compatible, "PrivacyEngine reports incompatible model, optimizer, or data loader."
print("PrivacyEngine compatibility passed.")

PrivacyEngine compatibility passed.


/usr/local/lib/python3.12/dist-packages/opacus/privacy_engine.py:98: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


In [22]:
captured_warning_rows = []

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    dp_model, dp_optimizer, dp_train_loader = privacy_engine.make_private_with_epsilon(
        module=dp_model,
        optimizer=dp_optimizer,
        criterion=criterion,
        data_loader=dp_train_loader,
        target_epsilon=TARGET_EPSILON,
        target_delta=DELTA,
        epochs=DP_EPOCHS,
        max_grad_norm=MAX_GRAD_NORM,
        poisson_sampling=POISSON_SAMPLING,
        clipping="flat",
        loss_reduction="mean",
    )

    for warning in caught:
        captured_warning_rows.append({
            "stage": "make_private_with_epsilon",
            "category": warning.category.__name__,
            "message": str(warning.message),
        })

noise_multiplier = float(dp_optimizer.noise_multiplier)
actual_max_grad_norm = float(dp_optimizer.max_grad_norm)
actual_sample_rate = float(
    getattr(dp_train_loader, "sample_rate", 1.0 / len(dp_train_loader))
)

print({
    "noise_multiplier": noise_multiplier,
    "max_grad_norm": actual_max_grad_norm,
    "sample_rate": actual_sample_rate,
    "delta": DELTA,
})

{'noise_multiplier': 0.469818115234375, 'max_grad_norm': 1.0, 'sample_rate': 0.002898550724637681, 'delta': 1.134031140495118e-05}


## 11. Train the private MLP and track epsilon

In [23]:
dp_history_rows = []
dp_started_at = time.time()

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")

    for epoch in range(1, DP_EPOCHS + 1):
        epoch_loss = train_one_epoch(
            dp_model,
            dp_train_loader,
            dp_optimizer,
            criterion,
            DEVICE,
        )

        epsilon_after_epoch = float(privacy_engine.get_epsilon(DELTA))

        dp_history_rows.append({
            "epoch": epoch,
            "train_loss": epoch_loss,
            "epsilon": epsilon_after_epoch,
            "delta": DELTA,
        })

        print(
            f"DP epoch {epoch:02d}/{DP_EPOCHS}: "
            f"loss={epoch_loss:.6f}, epsilon={epsilon_after_epoch:.4f}"
        )

    for warning in caught:
        captured_warning_rows.append({
            "stage": "dp_training",
            "category": warning.category.__name__,
            "message": str(warning.message),
        })

dp_training_seconds = time.time() - dp_started_at
actual_epsilon = float(privacy_engine.get_epsilon(DELTA))

dp_history = pd.DataFrame(dp_history_rows)
dp_history.to_csv(
    RESULTS_DIR / "dp_sgd_training_history.csv",
    index=False,
)

warnings_table = pd.DataFrame(
    captured_warning_rows,
    columns=["stage", "category", "message"],
)
warnings_table.to_csv(
    RESULTS_DIR / "opacus_warnings.csv",
    index=False,
)

assert np.isfinite(actual_epsilon)
assert actual_epsilon > 0
assert actual_epsilon <= TARGET_EPSILON + 0.10, (
    "Actual epsilon exceeded the target beyond the smoke-test tolerance."
)

print({
    "actual_epsilon": actual_epsilon,
    "target_epsilon": TARGET_EPSILON,
    "training_seconds": round(dp_training_seconds, 2),
})

DP epoch 01/5: loss=0.230249, epsilon=5.4083
DP epoch 02/5: loss=0.131198, epsilon=6.2579
DP epoch 03/5: loss=0.121218, epsilon=6.9217
DP epoch 04/5: loss=0.113391, epsilon=7.4903
DP epoch 05/5: loss=0.110352, epsilon=7.9986
{'actual_epsilon': 7.998628515798507, 'target_epsilon': 8.0, 'training_seconds': 19.71}


## 12. Evaluate the DP-SGD smoke model

In [24]:
dp_validation_probabilities = predict_probabilities(
    dp_model,
    validation_loader,
    DEVICE,
)
dp_test_probabilities = predict_probabilities(
    dp_model,
    test_loader,
    DEVICE,
)

dp_selected_threshold, dp_threshold_search = select_f2_threshold(
    validation_labels,
    dp_validation_probabilities,
)

dp_ids_results = pd.DataFrame([
    calculate_ids_metrics(
        "pytorch_dp_sgd_smoke",
        "target_validation",
        "default_0_5",
        validation_labels,
        dp_validation_probabilities,
        0.5,
    ),
    calculate_ids_metrics(
        "pytorch_dp_sgd_smoke",
        "KDDTest+",
        "default_0_5",
        test_labels,
        dp_test_probabilities,
        0.5,
    ),
    calculate_ids_metrics(
        "pytorch_dp_sgd_smoke",
        "KDDTest+",
        "validation_selected_F2",
        test_labels,
        dp_test_probabilities,
        dp_selected_threshold,
    ),
])

dp_threshold_search.to_csv(
    RESULTS_DIR / "dp_sgd_smoke_threshold_search.csv",
    index=False,
)
dp_ids_results.to_csv(
    RESULTS_DIR / "private_smoke_test.csv",
    index=False,
)

display(dp_ids_results)

,model,split,threshold_policy,threshold,accuracy,precision,recall,f1,fnr,fpr,roc_auc,pr_auc,tn,fp,fn,tp
0,pytorch_dp_sgd_smoke,target_validation,default_0_5,0.50,0.976185,0.989270,0.959236,0.974021,0.040764,0.009059,0.991163,0.993212,6673,61,239,5624
1,pytorch_dp_sgd_smoke,KDDTest+,default_0_5,0.50,0.733100,0.914699,0.585755,0.714170,0.414245,0.072186,0.897136,0.919280,9010,701,5316,7517
2,pytorch_dp_sgd_smoke,KDDTest+,validation_selected_F2,0.07,0.772268,0.914237,0.662043,0.767965,0.337957,0.082072,0.897136,0.919280,8914,797,4337,8496


In [25]:
non_private_tuned = non_private_results[
    non_private_results["threshold_policy"] == "validation_selected_F2"
].iloc[0]

dp_tuned = dp_ids_results[
    dp_ids_results["threshold_policy"] == "validation_selected_F2"
].iloc[0]

comparison_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "fnr",
    "fpr",
    "roc_auc",
    "pr_auc",
]

comparison_rows = []
for metric in comparison_metrics:
    comparison_rows.append({
        "metric": metric,
        "non_private_pytorch": float(non_private_tuned[metric]),
        "dp_sgd_smoke": float(dp_tuned[metric]),
        "dp_minus_non_private": float(dp_tuned[metric] - non_private_tuned[metric]),
    })

privacy_utility_smoke_comparison = pd.DataFrame(comparison_rows)
privacy_utility_smoke_comparison.to_csv(
    RESULTS_DIR / "dp_sgd_smoke_utility_comparison.csv",
    index=False,
)

display(privacy_utility_smoke_comparison)

,metric,non_private_pytorch,dp_sgd_smoke,dp_minus_non_private
0,accuracy,0.816537,0.772268,-0.044269
1,precision,0.958945,0.914237,-0.044708
2,recall,0.708018,0.662043,-0.045975
3,f1,0.814596,0.767965,-0.046630
4,fnr,0.291982,0.337957,0.045975
5,fpr,0.040058,0.082072,0.042014
6,roc_auc,0.910364,0.897136,-0.013228
7,pr_auc,0.935714,0.919280,-0.016434


## 13. Save model states, configuration, and reproducibility manifest

In [26]:
def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_module", model)

torch.save(
    non_private_model.state_dict(),
    MODEL_DIR / "non_private_pytorch_mlp_smoke.pt",
)
torch.save(
    unwrap_model(dp_model).state_dict(),
    MODEL_DIR / "dp_sgd_mlp_smoke.pt",
)

smoke_configuration = {
    "protocol_version": "ROADMAP_REVISED_V2",
    "experiment": "04_dp_sgd_feasibility_smoke_test",
    "seed": SEED,
    "device": str(DEVICE),
    "architecture": {
        "input_dim": INPUT_DIM,
        "hidden_dims": list(HIDDEN_DIMS),
        "activation": "ReLU",
        "output": "one logit",
        "batch_norm": False,
    },
    "training": {
        "optimizer": "Adam",
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "non_private_epochs": NON_PRIVATE_EPOCHS,
        "dp_epochs": DP_EPOCHS,
        "loss": "BCEWithLogitsLoss",
    },
    "dp": {
        "target_epsilon": TARGET_EPSILON,
        "actual_epsilon": actual_epsilon,
        "delta": DELTA,
        "noise_multiplier": noise_multiplier,
        "max_grad_norm": actual_max_grad_norm,
        "sample_rate": actual_sample_rate,
        "accountant": ACCOUNTANT,
        "secure_mode": SECURE_MODE,
        "poisson_sampling": POISSON_SAMPLING,
        "clipping": "flat",
    },
    "parity": {
        "passed": parity_pass,
        "tolerances": PARITY_TOLERANCES,
    },
}

with (RESULTS_DIR / "config.json").open("w") as file:
    json.dump(smoke_configuration, file, indent=2, default=str)

In [27]:
dp_smoke_manifest = {
    **smoke_configuration,
    "created_at_unix": time.time(),
    "dataset": {
        "train_file": str(TRAIN_FILE),
        "test_file": str(TEST_FILE),
        "train_sha256": actual_train_hash,
        "test_sha256": actual_test_hash,
    },
    "split": {
        "target_train_rows": len(target_train_indices),
        "target_validation_rows": len(target_validation_indices),
        "shadow_pool_rows": len(shadow_pool_indices),
    },
    "fixed_preprocessing": {
        "path": str(PREPROCESSOR_FILE),
        "scope": "Privacy accounting is conditional on this fixed preprocessing.",
    },
    "accepted_baseline": {
        "manifest_path": str(BASELINE_MANIFEST_FILE),
        "results_path": str(BASELINE_RESULTS_FILE),
    },
    "runtime": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "sklearn": sklearn.__version__,
        "torch": torch.__version__,
        "opacus": __import__("opacus").__version__,
    },
    "outputs": {
        "parity_csv": str(RESULTS_DIR / "non_private_pytorch_parity.csv"),
        "private_smoke_csv": str(RESULTS_DIR / "private_smoke_test.csv"),
        "warnings_csv": str(RESULTS_DIR / "opacus_warnings.csv"),
    },
}

manifest_output_file = RESULTS_DIR / "dp_sgd_smoke_manifest.json"

with manifest_output_file.open("w") as file:
    json.dump(dp_smoke_manifest, file, indent=2, default=str)

with (MANIFEST_DIR / "dp_sgd_smoke_manifest.json").open("w") as file:
    json.dump(dp_smoke_manifest, file, indent=2, default=str)

print("Manifest saved:", manifest_output_file)

Manifest saved: /content/drive/MyDrive/ML-DP-NID/results/dp_sgd_smoke_test/dp_sgd_smoke_manifest.json


## 14. Final feasibility gate

In [28]:
required_output_files = [
    RESULTS_DIR / "non_private_pytorch_parity.csv",
    RESULTS_DIR / "non_private_pytorch_ids_results.csv",
    RESULTS_DIR / "private_smoke_test.csv",
    RESULTS_DIR / "dp_sgd_training_history.csv",
    RESULTS_DIR / "opacus_warnings.csv",
    RESULTS_DIR / "dp_sgd_smoke_manifest.json",
]

missing_outputs = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

epsilon_valid = (
    np.isfinite(actual_epsilon)
    and actual_epsilon > 0
    and actual_epsilon <= TARGET_EPSILON + 0.10
)

feasibility_pass = bool(
    parity_pass
    and is_compatible
    and not compatibility_errors
    and epsilon_valid
    and not missing_outputs
)

print({
    "pytorch_parity_pass": parity_pass,
    "opacus_compatible": is_compatible,
    "actual_epsilon": actual_epsilon,
    "delta": DELTA,
    "all_outputs_saved": not missing_outputs,
    "experiment_04_feasibility_pass": feasibility_pass,
})

if not feasibility_pass:
    raise RuntimeError(
        "Experiment 04 feasibility gate failed. Do not start the DP-SGD sweep."
    )

print(
    "\nExperiment 04 feasibility gate passed. "
    "Upload the requested result files for supervisory review before Experiment 05."
)

{'pytorch_parity_pass': True, 'opacus_compatible': True, 'actual_epsilon': 7.998628515798507, 'delta': 1.134031140495118e-05, 'all_outputs_saved': True, 'experiment_04_feasibility_pass': True}

Experiment 04 feasibility gate passed. Upload the requested result files for supervisory review before Experiment 05.
